# ECB Communications → Supervised Learning (TF‑IDF) — Broad Filter & 3‑Class Labels

This notebook loads ECB statements and MRO policy rates, detects effective rate-change **events** from a daily series,
maps them to **policy‑relevant statements** within a business‑day window, and trains text models.

It includes:
- Robust CSV loading & date normalization
- **Broad policy filter** (title/url/category keywords, multi-language hints)
- Event → statement mapping with **wider window** (default `[-5BD, +3BD]`)
- **Three‑class labeling** (`−1`/`0`/`+1`) so you can train even if there are few rate changes

> Adjust the file paths in the first code cell if needed.


In [1]:
# --- Setup
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.tseries.offsets import BDay

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.metrics import classification_report, confusion_matrix

# Optional helpers; minimal fallbacks if not available
try:
    from skfin.text import coefs_plot, show_text
    from skfin.plot import line
    _HAS_SKFIN = True
except Exception:
    _HAS_SKFIN = False
    def coefs_plot(df_coef, title="Coefficients"):
        s = df_coef.squeeze()
        top = s.nlargest(10)
        bot = s.nsmallest(10)
        fig, ax = plt.subplots(figsize=(8, 6))
        both = pd.concat([bot, top])
        both.sort_values().plot(kind="barh", ax=ax)
        ax.set_title(title)
        plt.tight_layout()

    def show_text(df_text, lexica=None, n=None):
        for idx, row in df_text.iterrows():
            print(f"=== {idx} ===")
            txt = str(row['text'])
            print(txt[:800] + ("..." if len(txt) > 800 else ""))
            if lexica:
                print("\nTop positive:\n", list(lexica.get("positive", []).index))
                print("Top negative:\n", list(lexica.get("negative", []).index))
            print()

    def line(df, sort=False, ax=None, title=None):
        ax = ax or plt.gca()
        df.plot(ax=ax)
        if title:
            ax.set_title(title)

# Paths: try absolute then fallback to ./data
ABS_DATA = Path(r"C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset")
REL_DATA = Path("./data")
DATA_DIR = ABS_DATA if ABS_DATA.exists() else REL_DATA

ECB_STATEMENTS_CSV = DATA_DIR / "ecb_speeches_clean_minimal.csv"
ECB_POLICY_RATES_CSV = DATA_DIR / "ecb_policy_rates_daily_fake.csv"   # replace with your real file if needed
EUROSTOXX_CSV       = DATA_DIR / "eurostoxx_daily_fake.csv"           # optional

print('[PATHS]')
print('DATA_DIR         :', DATA_DIR)
print('STATEMENTS exists:', ECB_STATEMENTS_CSV.exists())
print('MRO exists       :', ECB_POLICY_RATES_CSV.exists())
print('EUROSTOXX exists :', EUROSTOXX_CSV.exists())

[PATHS]
DATA_DIR         : C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset
STATEMENTS exists: True
MRO exists       : True
EUROSTOXX exists : True


In [2]:
# --- Robust loaders & utilities
import csv

def _normalize_index_to_date_index(df, date_col='date'):
    """Normalize to a tz-naive date index with unique, sorted dates."""
    if date_col in df.columns:
        s = pd.to_datetime(df[date_col], errors='coerce')
    else:
        s = pd.to_datetime(df.index, errors='coerce')
    s = s.dt.tz_localize(None).dt.normalize()
    df = df.copy()
    df.index = s
    df = df[~df.index.isna()].sort_index()
    df = df[~df.index.duplicated(keep='first')]
    return df

def _sniff_csv_params(path, sample_bytes=65536):
    with open(path, 'rb') as f:
        raw = f.read(sample_bytes)
    encoding = 'utf-8-sig' if raw.startswith(b'\xef\xbb\xbf') else 'utf-8'
    try:
        text = raw.decode(encoding, errors='replace')
        dialect = csv.Sniffer().sniff(text, delimiters=[',',';','\t','|'])
        sep = dialect.delimiter
        quotechar = dialect.quotechar if dialect.quotechar else '"'
    except Exception:
        sep, quotechar = None, '"'
    return encoding, sep, quotechar

def load_ecb_statements(path=ECB_STATEMENTS_CSV):
    print(f"\n[LOAD] ECB statements from: {path}")
    enc, sep, quotechar = _sniff_csv_params(path)
    print(f"[SNIFF] encoding={enc} | sep={'auto' if sep is None else repr(sep)} | quotechar={repr(quotechar)}")
    tries = [
        dict(encoding=enc, sep=sep, engine='python', quotechar=quotechar, escapechar='\\'),
        dict(encoding=enc, sep=',', engine='python', quotechar='"', escapechar='\\'),
        dict(encoding=enc, sep=';', engine='python', quotechar='"', escapechar='\\'),
        dict(encoding='latin-1', sep=sep, engine='python', quotechar=quotechar, escapechar='\\'),
    ]
    df = None; last_err = None
    for i, kw in enumerate(tries, 1):
        try:
            df = pd.read_csv(path, **{k:v for k,v in kw.items() if v is not None})
            print(f"[READ OK] try#{i} -> shape={df.shape}")
            break
        except Exception as e:
            print(f"[READ FAIL] try#{i}: {e}")
            last_err = e
    if df is None:
        raise last_err
    if 'date' not in df.columns or 'text' not in df.columns:
        raise AssertionError("Statements CSV must contain 'date' and 'text'.")
    df = _normalize_index_to_date_index(df, 'date')
    df = df[['text']]
    print(f"[OK] statements: shape={df.shape} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def load_ecb_mro(path=ECB_POLICY_RATES_CSV):
    print(f"\n[LOAD] ECB MRO from: {path}")
    df = pd.read_csv(path)
    assert 'date' in df.columns and 'mro_rate' in df.columns, "MRO CSV must have 'date' and 'mro_rate'"
    df['mro_rate'] = pd.to_numeric(df['mro_rate'], errors='coerce')
    df = df.dropna(subset=['mro_rate'])
    df = _normalize_index_to_date_index(df, 'date')
    print(f"[OK] MRO daily: shape={df.shape} | unique rates={df['mro_rate'].nunique()} | span={df.index.min().date()}→{df.index.max().date()}")
    return df

def compute_rate_change_events(mro: pd.DataFrame):
    """Detect effective change dates (delta != 0). Return DataFrame 'events' with 'change' in {-1,+1}."""
    mro = mro.sort_index().copy()
    delta = mro['mro_rate'].diff()
    ev = delta[delta.fillna(0) != 0]
    events = pd.DataFrame(index=ev.index)
    events['change'] = np.sign(ev.values).astype(int)
    print(f"[EVENTS] changes: total={len(events)} | hikes={(events['change']==1).sum()} | cuts={(events['change']==-1).sum()}")
    return events

In [3]:
# --- Broad policy filter (title/url/category, multi-language hints)
def subset_policy_statements_broad(statements_csv_path, events_index):
    df = pd.read_csv(statements_csv_path)
    assert "date" in df.columns and "text" in df.columns, "CSV must have 'date' and 'text'"
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.tz_localize(None).dt.normalize()
    df = df.dropna(subset=["date"])

    title = df["title"].astype(str).str.lower() if "title" in df.columns else pd.Series("", index=df.index)
    url   = df["url"].astype(str).str.lower()   if "url"   in df.columns else pd.Series("", index=df.index)
    cat   = df["category"].astype(str).str.lower() if "category" in df.columns else pd.Series("", index=df.index)

    patt = (
        r"monetary policy decision|introductory statement|press conference|press release|"
        r"governing council|interest rate|policy rate|refinancing rate|"
        r"compte rendu|déclaration introductive|conférence de presse|"
        r"conferenza stampa|dichiarazione introduttiva|"
        r"pressekonferenz|einleitende erklärung|"
        r"/press/pr/|/press/pressconf/|/press/govcdec/"
    )
    keep = title.str.contains(patt, regex=True) | url.str_contains(patt, regex=True) if hasattr(url, "str") else title.str.contains(patt, regex=True)
    # add category if present
    try:
        keep = keep | cat.str.contains(patt, regex=True)
    except Exception:
        pass

    if keep.sum() == 0:
        # If nothing matched, keep all within event span so pipeline can proceed
        keep = pd.Series(True, index=df.index)

    df = df[keep].copy()

    # Move weekends to previous BD
    weekend = df["date"].dt.dayofweek >= 5
    if weekend.any():
        df.loc[weekend, "date"] = (df.loc[weekend, "date"] - BDay(1)).dt.normalize()

    df = df.sort_values("date").drop_duplicates("date", keep="first").set_index("date")
    df = df[["text"]]

    if len(events_index):
        start, end = events_index.min(), events_index.max()
        df = df.loc[(df.index >= start - BDay(10)) & (df.index <= end + BDay(10))]

    print(f"[FILTER-BROAD] policy-like statements kept: {len(df)}")
    return df

In [4]:
# --- Event→Statement mapping with wider window
def map_events_to_statements(statements: pd.DataFrame,
                             events: pd.DataFrame,
                             back_bdays=5, fwd_bdays=3, prefer_past=True):
    s_idx = pd.DatetimeIndex(statements.index).sort_values()
    ev_idx = pd.DatetimeIndex(events.index).sort_values()
    chosen = {}
    for e in ev_idx:
        window = set([e])
        for k in range(1, back_bdays+1):
            window.add(e - BDay(k))
        for k in range(1, fwd_bdays+1):
            window.add(e + BDay(k))
        candidates = s_idx.intersection(pd.DatetimeIndex(sorted(window)))
        if len(candidates) == 0:
            continue
        def bd_dist(s, e):
            if s == e: return 0
            k = 0; cur = s
            if s < e:
                while cur < e: cur += BDay(1); k += 1
            else:
                while cur > e: cur -= BDay(1); k += 1
            return k
        bestS, bestKey = None, None
        for s in candidates:
            dist = bd_dist(s, e)
            tie  = 1 if (prefer_past and s <= e) else 0
            key  = (dist, -tie)
            if bestKey is None or key < bestKey:
                bestKey, bestS = key, s
        change = int(events.loc[e, 'change'])
        prev = chosen.get(bestS)
        if prev is None or bestKey < prev[:2]:
            chosen[bestS] = (bestKey[0], bestKey[1], change)
    if not chosen:
        print(f"[MAP] No event mapped with window [-{back_bdays}BD, +{fwd_bdays}BD].")
        return statements.iloc[0:0].copy()
    S_dates = sorted(chosen.keys())
    df_lbl = statements.loc[S_dates].copy()
    df_lbl['change'] = [chosen[s][2] for s in S_dates]
    vc = df_lbl['change'].value_counts().to_dict()
    print(f"[MAP] labeled={len(df_lbl)} / {len(statements)} ({len(df_lbl)/len(statements)*100:.1f}%) | class_balance={vc}")
    return df_lbl

In [5]:
# --- 3-class labels (−1, 0, +1): no-change for policy days without a nearby event
def build_three_class_labels(statements_pol: pd.DataFrame, events: pd.DataFrame,
                             back_bdays=5, fwd_bdays=3, prefer_past=True):
    labeled_pm = map_events_to_statements(statements_pol, events, back_bdays, fwd_bdays, prefer_past)
    out = statements_pol.copy()
    out['change_3cls'] = 0
    if len(labeled_pm):
        out.loc[labeled_pm.index, 'change_3cls'] = labeled_pm['change'].astype(int)
    print(f"[3-CLASS] total={len(out)} | counts={out['change_3cls'].value_counts().to_dict()}")
    return out

In [6]:
# --- Models
def train_tfidf_logistic_binary(df_labeled: pd.DataFrame):
    X, y = df_labeled['text'], df_labeled['change']
    if y.nunique() < 2 or len(y) < 10:
        print('[BINARY MODEL] Not enough labeled samples.')
        return None, None
    est = Pipeline(steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=8000, lowercase=True,
                                  strip_accents='unicode', stop_words='english',
                                  token_pattern=r'\b[a-zA-Z]{3,}\b')),
        ('log1p', FunctionTransformer(np.log1p, validate=False)),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                                   class_weight='balanced', max_iter=5000, C=1.0,
                                   n_jobs=-1, random_state=0)),
    ])
    est.fit(X, y)
    vocab = np.array(sorted(est.named_steps['tfidf'].vocabulary_, key=lambda k: est.named_steps['tfidf'].vocabulary_[k]))
    coefs = est.named_steps['clf'].coef_[0]
    coef_df = pd.DataFrame(coefs, index=vocab, columns=['coef']).sort_values('coef')
    print('[BINARY MODEL] Logistic trained. n_features:', len(vocab))
    return est, coef_df

def train_tfidf_logistic_three_class(df_3: pd.DataFrame):
    X, y = df_3['text'], df_3['change_3cls']
    if y.nunique() < 2 or len(y) < 25:
        print('[3-CLASS MODEL] Not enough samples.')
        return None, None
    est = Pipeline(steps=[
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=8000, lowercase=True,
                                  strip_accents='unicode', stop_words='english',
                                  token_pattern=r'\b[a-zA-Z]{3,}\b')),
        ('clf', LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5,
                                   class_weight='balanced', max_iter=5000, C=1.0,
                                   n_jobs=-1, random_state=0)),
    ])
    est.fit(X, y)
    print('[3-CLASS MODEL] Trained on', len(X), 'policy statements.')
    return est, None

In [7]:
# --- Run end-to-end
statements_all = load_ecb_statements()
mro_daily      = load_ecb_mro()
events         = compute_rate_change_events(mro_daily)

# Broad policy filter
statements_pol = subset_policy_statements_broad(ECB_STATEMENTS_CSV, events.index)

# Event→statement mapping (±1 labels). Wider window by default.
df_labeled_pm = map_events_to_statements(statements_pol, events, back_bdays=5, fwd_bdays=3, prefer_past=True)

print('\n[SUMMARY — binary candidate]')
print('  policy-like      :', len(statements_pol))
print('  labeled (±1)     :', len(df_labeled_pm))
print('  unlabeled        :', len(statements_pol) - len(df_labeled_pm))

# Try binary model if enough examples
bin_est, bin_coef = train_tfidf_logistic_binary(df_labeled_pm)
if bin_coef is not None:
    coefs_plot(bin_coef.head(20), title='Binary model: top negative coefficients')
    coefs_plot(bin_coef.tail(20), title='Binary model: top positive coefficients')

# Always build 3-class dataset (−1/0/+1) to ensure we can train something
df_3 = build_three_class_labels(statements_pol, events, back_bdays=5, fwd_bdays=3, prefer_past=True)

tri_est, _ = train_tfidf_logistic_three_class(df_3)


[LOAD] ECB statements from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_speeches_clean_minimal.csv
[SNIFF] encoding=utf-8 | sep=',' | quotechar='"'
[READ OK] try#1 -> shape=(2939, 5)
[OK] statements: shape=(2249, 1) | span=1997-02-07→2025-09-30

[LOAD] ECB MRO from: C:\Users\Garance Latieule\analyzing ECB\Analyzing-ECB\dataset\ecb_policy_rates_daily_fake.csv
[OK] MRO daily: shape=(2827, 2) | unique rates=6 | span=2015-01-01→2025-10-31
[EVENTS] changes: total=11 | hikes=8 | cuts=3


AttributeError: 'Series' object has no attribute 'str_contains'